# The diet problem: the first LP, and the direction of an inequality

Six foods, each with a price per serving. Four nutrients, each with a daily minimum. A table saying
how much of each nutrient a serving of each food delivers. Find the cheapest basket that meets every
minimum.

This is the first LP most people meet, and it is small enough to hold in your head: one variable per
food, one constraint per nutrient, one line for the cost. The formulation is the whole lesson. In
particular, **every constraint has a direction, and the solver will honour the one you typed** — it
has no way of knowing the one you meant.

## Setup: where the package lives

This notebook builds its models by hand and then checks them against `orteach`, the package in
`src/`. Run from a clone of the repository, `../../src` is right there. On Colab there is no clone
until this cell makes one, and no `gurobipy` until it installs it. Nothing here needs a secret.

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/sear-labs/teaching-code"

try:
    import google.colab                      # noqa: F401 - succeeds only on Colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if not os.path.isdir("/content/teaching-code"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/teaching-code"], check=True)
    os.chdir("/content/teaching-code/notebooks/01_lp_formulation")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gurobipy>=11,<14"], check=True)

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
try:
    import orteach                            # noqa: F401
except ImportError:
    raise SystemExit("orteach not found: run this notebook from its own folder inside the repository, "
                     "so that ../../src exists.")
root = os.path.abspath(os.path.join("..", ".."))
print("package:", os.path.relpath(os.path.dirname(orteach.__file__), root))

package: src\orteach


## Licence setup

Nothing here needs a key: `pip install gurobipy` ships a size-limited licence and the models below
sit well inside it. A machine with its own licence file uses that instead, and on Colab three
secrets read from the key icon in the left sidebar are used when they are there — three named here,
none contained. The environment starts silent, so no licence number lands in an output cell.

In [2]:
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
        licence = "Colab Secrets (WLS)"
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        licence = "the size-limited licence pip ships"     # no key needed; see the note above
except ImportError:
    licence = "local gurobi.lic"
env.start()
print("licence:", licence)

licence: local gurobi.lic


## The instance is a table

Costs, minimums and a food-by-nutrient grid: instance data indexed by the model's own sets, named
nowhere in the prose. So it lives in `data/raw/` and both this notebook and the package read the same
two files.

The model will look values up by `(food, nutrient)`, so the dictionary form is printed as well as the
grid — the key is the thing every constraint below is built from.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach import tolerance
from orteach.diet import load_diet

inst = load_diet("macros")

print(f"{'food':8} {'$/serv':>7} " + " ".join(f"{n:>9}" for n in inst.nutrients))
for f in inst.foods:
    print(f"{f:8} {inst.cost[f]:7.2f} " + " ".join(f"{inst.content[f, n]:9.1f}" for n in inst.nutrients))
print(f"{'minimum':8} {'':7} " + " ".join(f"{inst.minimum[n]:9.1f}" for n in inst.nutrients))
print()
print("content[('milk', 'protein')] =", inst.content["milk", "protein"])
print("minimum['calories']          =", inst.minimum["calories"])

# to try a different price, edit the loaded table and the same value flows into both the
# hand-built model and the package check at the bottom:
# inst.cost["cheese"] = 4.00

food      $/serv   protein       fat     carbs  calories
bread       2.00       4.0       1.0      15.0      90.0
milk        3.50       8.0       5.0      11.7     120.0
cheese      8.00       7.0       9.0       0.4     106.0
potato      1.50       1.3       0.1      22.6      97.0
fish       11.00       8.0       7.0       0.0     130.0
yogurt      1.00       9.2       1.0      17.0     180.0
minimum               10.0       8.0      10.0     300.0

content[('milk', 'protein')] = 8.0
minimum['calories']          = 300.0


## Predict before building anything

Look at the table. For each nutrient, which food delivers it most cheaply per unit? Write down the two
or three foods you expect the cheapest basket to be made of — and one you expect it never to touch.

## The model, and one variable per food

Servings are continuous and cannot be negative; nothing else is known about them yet.

In [4]:
m = gp.Model("diet", env=env)
tolerance.apply(m)

x = m.addVars(inst.foods, lb=0.0, name="servings")
print(len(x), "variables:", list(x.keys()))

6 variables: ['bread', 'milk', 'cheese', 'potato', 'fish', 'yogurt']


## The objective is the grocery bill

Cost per serving times servings, summed over foods. Minimised.

In [5]:
m.setObjective(gp.quicksum(inst.cost[f] * x[f] for f in inst.foods), GRB.MINIMIZE)
m.update()
print(m.getObjective())

2.0 servings[bread] + 3.5 servings[milk] + 8.0 servings[cheese] + 1.5 servings[potato] + 11.0 servings[fish] + servings[yogurt]


## One constraint per nutrient — written out once

The protein row, longhand: protein per serving times servings, summed over foods, **at least** the
minimum. The `>=` is the decision in this line. Say to yourself why it is not `<=`. Gurobi reports
the sense back as a single character — `>` for `>=`.

In [6]:
protein = m.addConstr(
    gp.quicksum(inst.content[f, "protein"] * x[f] for f in inst.foods) >= inst.minimum["protein"],
    name="nutrient[protein]")
m.update()
print(m.getRow(protein), protein.Sense, protein.RHS)

4.0 servings[bread] + 8.0 servings[milk] + 7.0 servings[cheese] + 1.3 servings[potato] + 8.0 servings[fish] + 9.2 servings[yogurt] > 10.0


The other three rows have identical shape, so a loop is honest here — it repeats a step you have
already seen, it does not hide a decision.

In [7]:
rows = {"protein": protein}
for n in ["fat", "carbs", "calories"]:
    rows[n] = m.addConstr(
        gp.quicksum(inst.content[f, n] * x[f] for f in inst.foods) >= inst.minimum[n],
        name=f"nutrient[{n}]")
m.update()
print(m.NumConstrs, "constraints")

4 constraints


## Read back what you built

Before solving, print the constraints the model actually holds — name, sense, right-hand side. Read it
the way you would if you suspected a typo somewhere in the model: which column would you check first?

In [8]:
for c in m.getConstrs():
    print(f"{c.ConstrName:20} sense {c.Sense}  rhs {c.RHS:6.1f}    {m.getRow(c)}")

nutrient[protein]    sense >  rhs   10.0    4.0 servings[bread] + 8.0 servings[milk] + 7.0 servings[cheese] + 1.3 servings[potato] + 8.0 servings[fish] + 9.2 servings[yogurt]
nutrient[fat]        sense >  rhs    8.0    servings[bread] + 5.0 servings[milk] + 9.0 servings[cheese] + 0.1 servings[potato] + 7.0 servings[fish] + servings[yogurt]
nutrient[carbs]      sense >  rhs   10.0    15.0 servings[bread] + 11.7 servings[milk] + 0.4 servings[cheese] + 22.6 servings[potato] + 17.0 servings[yogurt]
nutrient[calories]   sense >  rhs  300.0    90.0 servings[bread] + 120.0 servings[milk] + 106.0 servings[cheese] + 97.0 servings[potato] + 130.0 servings[fish] + 180.0 servings[yogurt]


## Two side conditions, as bounds rather than rows

The course version of this problem insists on at least half a serving of fish and at most one serving
of milk. Each involves one variable, so each is a **bound** on that variable, not a constraint row —
the solver treats bounds more cheaply and the model reads more honestly.

In [9]:
FISH_MIN = 0.5      # servings, at least
MILK_MAX = 1.0      # servings, at most

x["fish"].LB = FISH_MIN
x["milk"].UB = MILK_MAX
m.update()
print("fish bounds", (x["fish"].LB, x["fish"].UB), "   milk bounds", (x["milk"].LB, x["milk"].UB))

fish bounds (0.5, inf)    milk bounds (0.0, 1.0)


## Solve

Predict the bill to within a dollar before running this, and which of the four minimums will be met
exactly.

In [10]:
m.optimize()
print(f"\ncheapest basket: ${m.ObjVal:.2f} per day")


cheapest basket: $8.89 per day


## The basket, and which minimums bind

A nutrient whose intake sits exactly on its minimum is one the solver had to work for. The others
came along for free with the foods chosen for the binding ones.

In [11]:
print(f"{'food':8} {'servings':>9}")
for f in inst.foods:
    if x[f].X > tolerance.FEASIBILITY_ATOL:
        print(f"{f:8} {x[f].X:9.3f}")
print()
print(f"{'nutrient':10} {'minimum':>8} {'intake':>8}   binding?")
for n in inst.nutrients:
    got = sum(inst.content[f, n] * x[f].X for f in inst.foods)
    tight = abs(got - inst.minimum[n]) < tolerance.FEASIBILITY_ATOL
    print(f"{n:10} {inst.minimum[n]:8.1f} {got:8.2f}   {'yes' if tight else ''}")

food      servings
milk         0.737
fish         0.500
yogurt       0.814

nutrient    minimum   intake   binding?
protein        10.0    17.39   
fat             8.0     8.00   yes
carbs          10.0    22.46   
calories      300.0   300.00   yes


Compare with what you wrote down. Which foods surprised you, and does the binding column explain them?

---

## The same model, one character different

Three term folders of this course shipped this exact instance with the protein row written `<=`.
Everything else was identical. Here is that model, built the same way, so you can see what the solver
made of it.

In [12]:
m_flip = gp.Model("diet, protein row reversed", env=env)
tolerance.apply(m_flip)
xf = m_flip.addVars(inst.foods, lb=0.0, name="servings")
xf["fish"].LB = FISH_MIN
xf["milk"].UB = MILK_MAX
m_flip.setObjective(gp.quicksum(inst.cost[f] * xf[f] for f in inst.foods), GRB.MINIMIZE)

rows_flip = {}
rows_flip["protein"] = m_flip.addConstr(
    gp.quicksum(inst.content[f, "protein"] * xf[f] for f in inst.foods) <= inst.minimum["protein"],
    name="nutrient[protein]")                                            # <-- the one character
for n in ["fat", "carbs", "calories"]:
    rows_flip[n] = m_flip.addConstr(
        gp.quicksum(inst.content[f, n] * xf[f] for f in inst.foods) >= inst.minimum[n],
        name=f"nutrient[{n}]")
m_flip.update()
print(m_flip.NumVars, "variables,", m_flip.NumConstrs, "constraints — same counts as before")

6 variables, 4 constraints — same counts as before


Before you run it: will this bill be higher or lower than the one above, and why? The feasible
regions are different, not nested, so the answer is not automatic.

In [13]:
m_flip.optimize()
print(f"\nbasket with protein reversed: ${m_flip.ObjVal:.2f} per day   (correct model: ${m.ObjVal:.2f})")
print()
print(f"{'food':8} {'servings':>9}")
for f in inst.foods:
    if xf[f].X > tolerance.FEASIBILITY_ATOL:
        print(f"{f:8} {xf[f].X:9.3f}")
print()
print(f"{'nutrient':10} {'minimum':>8} {'intake':>8}")
for n in inst.nutrients:
    got = sum(inst.content[f, n] * xf[f].X for f in inst.foods)
    print(f"{n:10} {inst.minimum[n]:8.1f} {got:8.2f}")


basket with protein reversed: $12.08 per day   (correct model: $8.89)

food      servings
milk         0.054
cheese       0.449
potato       1.865
fish         0.500

nutrient    minimum   intake
protein        10.0    10.00
fat             8.0     8.00
carbs          10.0    42.96
calories      300.0   300.00


Look at the protein line. The model is sitting exactly on 10 — from which side? What question did the
solver think it was answering, and why did the bill go the way it went?

## Where the evidence was

The original notebooks printed their constraint table after solving. Here is the same table for the
reversed model. The evidence is one character wide.

In [14]:
for c in m_flip.getConstrs():
    print(f"{c.ConstrName:20} sense {c.Sense}  rhs {c.RHS:6.1f}")

nutrient[protein]    sense <  rhs   10.0
nutrient[fat]        sense >  rhs    8.0
nutrient[carbs]      sense >  rhs   10.0
nutrient[calories]   sense >  rhs  300.0


---

## The same formulation on a different table

The graduate section used a different instance — four foods, four vitamins — with the same
formulation. Load it and the code above is unchanged; only the sets are different. That is what
keeping the table out of the notebook buys.

In [15]:
vit = load_diet("vitamins")

print(f"{'food':8} {'$/serv':>7} " + " ".join(f"{n:>10}" for n in vit.nutrients))
for f in vit.foods:
    print(f"{f:8} {vit.cost[f]:7.2f} " + " ".join(f"{vit.content[f, n]:10.0f}" for n in vit.nutrients))
print(f"{'minimum':8} {'':7} " + " ".join(f"{vit.minimum[n]:10.0f}" for n in vit.nutrients))

food      $/serv  vitamin_a  vitamin_c  vitamin_d       iron
milk        3.00       6400         40        540         28
tuna        2.70        237          0          0          7
bread       1.80          0          0          0         13
spinach     2.16      34000         71          0          8
minimum                5000         75        400         12


Same three steps — variables, objective, one `>=` row per nutrient — with no side bounds this time.
Predict which foods the basket will use.

In [16]:
m2 = gp.Model("diet, vitamins", env=env)
tolerance.apply(m2)
x2 = m2.addVars(vit.foods, lb=0.0, name="servings")
m2.setObjective(gp.quicksum(vit.cost[f] * x2[f] for f in vit.foods), GRB.MINIMIZE)
rows2 = {n: m2.addConstr(gp.quicksum(vit.content[f, n] * x2[f] for f in vit.foods) >= vit.minimum[n],
                         name=f"nutrient[{n}]") for n in vit.nutrients}
m2.optimize()

print(f"\ncheapest basket: ${m2.ObjVal:.2f} per day")
for f in vit.foods:
    if x2[f].X > tolerance.FEASIBILITY_ATOL:
        print(f"  {f:8} {x2[f].X:7.3f} servings")


cheapest basket: $3.60 per day
  milk       0.741 servings
  spinach    0.639 servings


---

# Now the streamlined version

Three models built from the same three steps, so they belong in one function now. The package solver
takes the instance as an argument, takes the side bounds as named arguments, and keeps the reversed
row callable as `flip=` — not as a feature, but so that the mistake stays reproducible.

In [17]:
from orteach import diet
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg      = diet.solve(inst, lower={"fish": FISH_MIN}, upper={"milk": MILK_MAX}, env=env)
pkg_flip = diet.solve(inst, lower={"fish": FISH_MIN}, upper={"milk": MILK_MAX}, flip=("protein",), env=env)
pkg_vit  = diet.solve(vit, env=env)

for b in (pkg, pkg_flip, pkg_vit):
    print(f"{b.label:34} ${b.objective:7.2f}   {b.chosen()}")

diet_macros                        $   8.89   ['fish', 'milk', 'yogurt']
diet_macros (flipped: protein)     $  12.08   ['cheese', 'fish', 'milk', 'potato']
diet_vitamins                      $   3.60   ['milk', 'spinach']


## The agreement assertion

Each hand-built model against the package, number by number: the bill and every serving. Both sides
solved at the same tightened tolerances, so agreement to `AGREEMENT_RTOL` is a claim the computation
supports.

In [18]:
checks = [("macros bill", m.ObjVal, pkg.objective),
          ("reversed bill", m_flip.ObjVal, pkg_flip.objective),
          ("vitamins bill", m2.ObjVal, pkg_vit.objective)]
for f in inst.foods:
    checks.append((f"macros {f}", x[f].X, pkg.servings[f]))
    checks.append((f"reversed {f}", xf[f].X, pkg_flip.servings[f]))
for f in vit.foods:
    checks.append((f"vitamins {f}", x2[f].X, pkg_vit.servings[f]))

worst = max(rel_diff(h, p) for _, h, p in checks)
print(f"{len(checks)} comparisons")
for name, hand, packaged in checks[:3]:
    print(f"  {name:16} hand {hand:10.4f}   package {packaged:10.4f}   rel {rel_diff(hand, packaged):.2e}")
print("  ...")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

19 comparisons
  macros bill      hand     8.8942   package     8.8942   rel 0.00e+00
  reversed bill    hand    12.0813   package    12.0813   rel 0.00e+00
  vitamins bill    hand     3.6025   package     3.6025   rel 0.00e+00
  ...

notebook and package agree to 0.0e+00


---

## Where to take this next

- Drop the fish minimum (`lower={}`) and re-solve. What happens to fish, and what does that say about
  why the bound was there?
- Calories usually have a ceiling as well as a floor. Add a second calories row with `<=` and a limit
  of your choosing. Two rows on one nutrient — is that a problem, and what happens if the ceiling is
  below the floor?
- Nobody eats 0.8 of a serving. Make the servings integer and compare the bill. Then look up which
  notebook in this library is about what that change does to the solver.